# Avocados2026

In [1]:
# Import Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns


<class 'ModuleNotFoundError'>: No module named 'pandas'

In [ ]:
# Create timestamp and author 
from datetime import datetime
time_now=datetime.now().strftime('%m_%d_%Y_%I%M%p')
print("Current time:{}".format(time_now))
print("Noah Choi")

In [ ]:
# Read the dataset and store it as data2026
data2026=pd.read_csv('2026PartYear.csv', encoding='utf-8')

In [ ]:
# head shows top 5 rows
data2026.head()

In [2]:
import csv

with open('2025.csv', mode='r') as f:
    reader = csv.DictReader(f)
    data_2025 = list(reader)

# Calculate total volume for 'Total U.S.'
total_vol_2025 = sum(
    float(row['Total Bulk and Bags']) 
    for row in data_2025 
    if row['Geography'] == 'Total U.S.'
)

print(f"2025 Total U.S. Volume: {total_vol_2025:,.2f}")

2025 Total U.S. Volume: 2,986,017,472.42


In [3]:
import csv

def load_and_summarize(filename):
    with open(filename, mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        # Filter rows for 'Total U.S.'
        us_data = [row for row in reader if row['Geography'] == 'Total U.S.']
        
        # Calculate Total Volume and Total Revenue
        total_vol = sum(float(r['Total Bulk and Bags']) for r in us_data)
        total_rev = sum(float(r['ASP Current Year']) * float(r['Total Bulk and Bags']) for r in us_data)
        weighted_asp = total_rev / total_vol if total_vol else 0
        
        return total_vol, total_rev, weighted_asp

vol_2025, rev_2025, asp_2025 = load_and_summarize('2025.csv')
print(f"2025 Total U.S. Volume: {vol_2025:,.2f} lbs | Revenue: \({rev_2025:,.2f} | Weighted ASP:\){asp_2025:.4f}")

<unknown>:17: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<>:17: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<>:17: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<>:17: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<>:17: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<ipython-input-3-32f4d33e765f>:17: SyntaxWarning: "\(" is an i

2025 Total U.S. Volume: 2,986,017,472.42 lbs | Revenue: \(3,494,783,136.99 | Weighted ASP:\)1.1704


In [6]:
import pyarrow.csv as pv
import pyarrow.compute as pc

# Read CSV into PyArrow Table
table = pv.read_csv('2025.csv')

# Filter rows
mask = pc.equal(table['Geography'], 'Total U.S.')
filtered_table = table.filter(mask)

# Compute Volume Sum
total_volume = pc.sum(filtered_table['Total Bulk and Bags']).as_py()
print(f"PyArrow Total Volume: {total_volume:,.2f}")

PyArrow Total Volume: 2,986,017,472.42


In [9]:
import duckdb

# Query directly from CSV file
query = """
SELECT 
    Geography,
    SUM("Total Bulk and Bags") AS total_volume,
    SUM("ASP Current Year" * "Total Bulk and Bags") AS total_revenue,
    SUM("ASP Current Year" * "Total Bulk and Bags") / SUM("Total Bulk and Bags") AS weighted_asp
FROM '2025.csv'
WHERE Geography = 'Total U.S.'
GROUP BY Geography
"""

result = duckdb.sql(query).df()
print(result)

    Geography  total_volume  total_revenue  weighted_asp
0  Total U.S.  2.986017e+09   3.494783e+09      1.170383
